<a href="https://colab.research.google.com/github/jishnusathyan0418/AI-resume-Assistant/blob/main/AI_Reumse_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install required packages
!pip install transformers accelerate gradio -q

# Verify GPU is available
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import gradio as gr

# model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
model_id = "HuggingFaceH4/zephyr-7b-beta"

print("Loading model... Please wait...")

#Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)
# Fix for models without pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map = "auto",
    trust_remote_code=True)

#setup the chat pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Model loaded successfully! Ready to chat.")

def polish_resume(position_name, resume_text, polish_prompt=""):
    if polish_prompt and polish_prompt.strip():
        prompt_text = f"Given the {resume_text} and the position of {position_name}, please polish the resume based on the following feedback: {polish_prompt}"
    else:
        prompt_text = f"Given the {resume_text} and the position of {position_name}, please polish the resume."

    # Format as chat message
    message = [
        {"role":"user", "content": prompt_text}
    ]

    #Apply chat template
    formated_prompt = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True)

    #Generate response
    output = pipe(
        formated_prompt,
        max_new_tokens = 512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_full_text=False
    )

    response = output[0]['generated_text']
    return response




In [ ]:
#Create gradio interface

chat_application = gr.Interface(
    fn=polish_resume,
    flagging_mode='never',
       inputs=[
        gr.Textbox(label="Position Name", placeholder="Enter the name of the position..."),
        gr.Textbox(label="Resume Content", placeholder="Paste your resume content here...", lines=20),
        gr.Textbox(label="Polish Instruction (Optional)", placeholder="Enter specific instructions or areas for improvement (optional)...", lines=2),
    ],
    outputs=gr.Textbox(label="Polished Content", lines=5),
    title="Resume Polish Application",
    description="This application helps you polish your resume. Enter the position you want to apply, your resume content, and specific instructions or areas for improvement (optional), then get a polished version of your content."
)


chat_application.launch(share=True)